# Dataset Preparation for ML

In [ ]:
from IPython.display import display

from zebrafish_tensor_utils import (
    build_moa_labeled_tensor_dataset,
    build_tensor_embedding_2d,
    plot_tensor_embedding_2d,
)

from zebrafish_notebook_utils import (
    build_compound_image_condition_map,
    build_compound_image_run_map,
    configure_full_dataframe_display,
)

In [ ]:
configure_full_dataframe_display()

compound_image_map_df = build_compound_image_run_map()
condition_df = build_compound_image_condition_map(compound_image_map_df)
condition_df.shape

In [ ]:
# User inputs
# Choose four mechanisms arranged as two similar pairs.
# Pair 1: GABAAR_Antagonist vs GABAAR_NegativeAllostericModulator.
# Pair 2: NMDAR_Activation vs NMDAR_Antagonist.
# The two mechanisms within each pair act on the same receptor system, so they should be
# more similar to each other than to the mechanisms in the other pair.
# This gives a downstream ML problem with two local confusions but clearer separation across pairs.
selected_mechanisms = [
    "GABAAR_Antagonist",
    "GABAAR_NegativeAllostericModulator",
    "NMDAR_Activation",
    "NMDAR_Antagonist",
]
# Use all three treatment bands by default; change to ["high"] for a stricter setup.
selected_concentrations = ["high", "mid", "low"]
# Keep per-class and per-compound caps small initially to make iteration cheaper.
max_compounds_per_action = 2
max_tensors_per_compound = 2
# Tensor size is T x Z x Y x X after cached loading, downsampling, and drift correction.
output_size = (10, 1, 32, 32)
only_active = True
# XY-only augmentation: small rotations preserve anatomy while adding mild variability.
num_random_rotations = 2
rotation_range_degrees = 5.0
normalize_global_drift = True
loess_frac = 0.25
use_cache = True
use_tiff_cache = True
random_seed = 0
# PCA is always available through scikit-learn. UMAP needs the optional umap-learn package.
embedding_method = "pca"

In [ ]:
dataset = build_moa_labeled_tensor_dataset(
    condition_df=condition_df,
    selected_mechanisms=selected_mechanisms,
    selected_concentrations=selected_concentrations,
    max_compounds_per_action=max_compounds_per_action,
    max_tensors_per_compound=max_tensors_per_compound,
    output_size=output_size,
    only_active=only_active,
    num_random_rotations=num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    normalize_global_drift=normalize_global_drift,
    loess_frac=loess_frac,
    use_cache=use_cache,
    use_tiff_cache=use_tiff_cache,
    random_seed=random_seed,
)

In [ ]:
dataset["tensors"].shape, dataset["labels"].shape, dataset["label_map"]

In [ ]:
display(dataset["metadata"].head(5))
display(dataset["metadata"].groupby(["label", "label_name"]).size().reset_index(name="n_examples"))

In [ ]:
# Flatten each tensor example and reduce it to 2D for quick class-separation inspection.
# Water remains class 0 in both the dataset labels and the plotted legend.
embedding_df = build_tensor_embedding_2d(
    tensors=dataset["tensors"],
    labels=dataset["labels"],
    label_map=dataset["label_map"],
    metadata=dataset["metadata"],
    method=embedding_method,
    random_state=random_seed,
)
display(embedding_df.head(10))

In [ ]:
plot_tensor_embedding_2d(
    embedding_df,
    title=f"{embedding_method.upper()} tensor embedding | water=0",
);